In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# =========================================================
# 1. device 설정
# =========================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# =========================================================
# 2. 데이터 불러오기
# =========================================================
transform = transforms.ToTensor()

train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

# =========================================================
# 3. MLP 모델 정의
#    MNIST 이미지는 1x28x28 -> 784 차원으로 펼쳐서 사용
# =========================================================
class ClassificationMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(28 * 28, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 10)   # 10개 클래스
            # CrossEntropyLoss를 쓸 것이므로 softmax는 넣지 않음
        )

    def forward(self, x):
        x = x.view(x.size(0), -1)  # (batch, 1, 28, 28) -> (batch, 784)
        return self.model(x)

model = ClassificationMLP().to(device)
print(model)

# =========================================================
# 4. loss / optimizer
# =========================================================
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# =========================================================
# 5. 학습 함수
# =========================================================
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)

        logits = model(xb)
        loss = criterion(logits, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * xb.size(0)

        pred = logits.argmax(dim=1)
        correct += (pred == yb).sum().item()
        total += yb.size(0)

    avg_loss = total_loss / len(loader.dataset)
    accuracy = correct / total
    return avg_loss, accuracy

# =========================================================
# 6. 평가 함수
# =========================================================
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)

            logits = model(xb)
            loss = criterion(logits, yb)

            total_loss += loss.item() * xb.size(0)

            pred = logits.argmax(dim=1)
            correct += (pred == yb).sum().item()
            total += yb.size(0)

    avg_loss = total_loss / len(loader.dataset)
    accuracy = correct / total
    return avg_loss, accuracy

# =========================================================
# 7. 학습 루프
# =========================================================
num_epochs = 5

for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    test_loss, test_acc = evaluate(model, test_loader, criterion, device)

    print(
        f"Epoch [{epoch+1}/{num_epochs}] | "
        f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
        f"Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.4f}"
    )

# =========================================================
# 8. 예측 예시
# =========================================================
model.eval()
with torch.no_grad():
    sample_x, sample_y = next(iter(test_loader))
    sample_x = sample_x[:10].to(device)
    sample_y = sample_y[:10]

    logits = model(sample_x)
    pred = logits.argmax(dim=1).cpu()

print("\nSample predictions:")
for i in range(10):
    print(f"Pred: {pred[i].item()}, True: {sample_y[i].item()}")

Using device: cuda
ClassificationMLP(
  (model): Sequential(
    (0): Linear(in_features=784, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=128, bias=True)
    (3): ReLU()
    (4): Linear(in_features=128, out_features=10, bias=True)
  )
)


Epoch [1/5] | Train Loss: 0.3454, Train Acc: 0.9039 | Test Loss: 0.1542, Test Acc: 0.9545


Epoch [2/5] | Train Loss: 0.1329, Train Acc: 0.9598 | Test Loss: 0.1225, Test Acc: 0.9623


Epoch [3/5] | Train Loss: 0.0882, Train Acc: 0.9730 | Test Loss: 0.0919, Test Acc: 0.9719


Epoch [4/5] | Train Loss: 0.0625, Train Acc: 0.9808 | Test Loss: 0.0762, Test Acc: 0.9769


Epoch [5/5] | Train Loss: 0.0480, Train Acc: 0.9853 | Test Loss: 0.0800, Test Acc: 0.9739

Sample predictions:
Pred: 7, True: 7
Pred: 2, True: 2
Pred: 1, True: 1
Pred: 0, True: 0
Pred: 4, True: 4
Pred: 1, True: 1
Pred: 4, True: 4
Pred: 9, True: 9
Pred: 5, True: 5
Pred: 9, True: 9


In [3]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# =========================================================
# 1. device 설정
# =========================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# =========================================================
# 2. 데이터 불러오기
# =========================================================
data = fetch_california_housing()
X = data.data           # 입력 feature
y = data.target         # 집값(회귀 target)

# train / test 분리
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 입력 feature 정규화
x_scaler = StandardScaler()
X_train = x_scaler.fit_transform(X_train)
X_test = x_scaler.transform(X_test)

# target도 학습 안정성을 위해 스케일링
y_scaler = StandardScaler()
y_train = y_scaler.fit_transform(y_train.reshape(-1, 1))
y_test = y_scaler.transform(y_test.reshape(-1, 1))

# numpy -> torch tensor
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

# DataLoader
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# =========================================================
# 3. MLP 모델 정의
# =========================================================
class RegressionMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)   # 회귀는 보통 마지막 activation 없음
        )

    def forward(self, x):
        return self.model(x)

model = RegressionMLP(input_dim=X_train.shape[1]).to(device)
print(model)

# =========================================================
# 4. loss / optimizer
# =========================================================
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# =========================================================
# 5. 학습 함수
# =========================================================
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0

    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)

        pred = model(xb)
        loss = criterion(pred, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * xb.size(0)

    return total_loss / len(loader.dataset)

# =========================================================
# 6. 평가 함수
# =========================================================
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)

            pred = model(xb)
            loss = criterion(pred, yb)

            total_loss += loss.item() * xb.size(0)

    return total_loss / len(loader.dataset)

# =========================================================
# 7. 학습 루프
# =========================================================
num_epochs = 20

for epoch in range(num_epochs):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    test_loss = evaluate(model, test_loader, criterion, device)

    print(f"Epoch [{epoch+1}/{num_epochs}] | Train Loss: {train_loss:.4f} | Test Loss: {test_loss:.4f}")

# =========================================================
# 8. 예측 예시
# =========================================================
model.eval()
with torch.no_grad():
    sample_x = X_test[:5].to(device)
    sample_pred = model(sample_x).cpu()

# 스케일 복원
pred_original = y_scaler.inverse_transform(sample_pred.numpy())
true_original = y_scaler.inverse_transform(y_test[:5].numpy())

print("\nSample predictions:")
for i in range(5):
    print(f"Pred: {pred_original[i][0]:.3f}, True: {true_original[i][0]:.3f}")

Using device: cuda


RegressionMLP(
  (model): Sequential(
    (0): Linear(in_features=8, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=32, bias=True)
    (3): ReLU()
    (4): Linear(in_features=32, out_features=1, bias=True)
  )
)


Epoch [1/20] | Train Loss: 0.4886 | Test Loss: 0.3606


Epoch [2/20] | Train Loss: 0.3173 | Test Loss: 0.3086


Epoch [3/20] | Train Loss: 0.2878 | Test Loss: 0.2864


...

Epoch [16/20] | Train Loss: 0.2169 | Test Loss: 0.2291


Epoch [17/20] | Train Loss: 0.2175 | Test Loss: 0.2295


Epoch [18/20] | Train Loss: 0.2154 | Test Loss: 0.2242


Epoch [19/20] | Train Loss: 0.2128 | Test Loss: 0.2286


Epoch [20/20] | Train Loss: 0.2111 | Test Loss: 0.2220

Sample predictions:
Pred: 0.561, True: 0.477
Pred: 1.297, True: 0.458
Pred: 4.825, True: 5.000
Pred: 2.622, True: 2.186
Pred: 3.127, True: 2.780
